# M3L3 E12 — Orquestador con OpenAI Functions (Resolution)
### Módulo 3 · Lecture 3 · Sistemas Multiagente

## ¿Qué vas a aprender hoy?
- dejar que el modelo elija tools por descripción.
- Conectar el concepto con M3L2.
- Leer código pequeño con explicación previa.
- Interpretar resultados y checks.


## ¿Qué necesitás saber antes?

Venís de M3L2 con LangChain, LCEL, PromptTemplate, RAG con FAISS y memoria conversacional. En M3L3 usamos esas piezas para coordinar varios agentes.

> **Sistema multiagente:** arquitectura donde varias unidades especializadas colaboran bajo una política de coordinación.


## Instalación e imports

En un notebook productivo podrías instalar `langchain`, `langchain-openai` y `faiss-cpu`. Aquí usamos Python estándar para que el foco sea el diseño multiagente y no la API key.


In [ ]:
from typing import Callable, TypedDict, Literal
from dataclasses import dataclass, field
import json

print("Setup listo: usamos Python estándar para que el notebook pueda correr sin API key.")


## Sección 1 — Tools y descriptions

OpenAI Functions / tool use permite que el modelo elija una función. La descripción de la tool guía esa decisión.

| Manual | Tool choice |
|---|---|
| escribimos clasificador | el modelo elige tool |
| más control explícito | más flexibilidad lingüística |


Simulamos herramientas con una dataclass: nombre, función y descripción.


In [ ]:
@dataclass
class ToolSpec:
    name: str
    func: Callable[[str], str]
    description: str

def hr_agent(query: str) -> str: return "HRAgent: vacaciones, beneficios y licencias."
def tech_agent(query: str) -> str: return "TechAgent: VPN, contraseña, MFA y notebook."
def billing_agent(query: str) -> str: return "BillingAgent: facturas, reembolsos y pagos."
tools = [ToolSpec("HRAgent", hr_agent, "RR.HH. vacaciones beneficios licencias seguro"), ToolSpec("TechAgent", tech_agent, "soporte técnico VPN contraseña MFA notebook"), ToolSpec("BillingAgent", billing_agent, "facturas reembolsos pagos recibos")]


## Sección 2 — Elegir tool

`choose_tool` puntúa palabras de la consulta contra la descripción. Es una simulación simple del efecto de descriptions precisas.


In [ ]:
def choose_tool(query: str, available_tools: list[ToolSpec]) -> ToolSpec | None:
    text_words = set(query.lower().split())
    scored = []
    for tool in available_tools:
        desc_words = set(tool.description.lower().split())
        scored.append((len(text_words & desc_words), tool))
    score, tool = max(scored, key=lambda item: item[0])
    return tool if score > 0 else None

def orchestrator_run(query: str, available_tools: list[ToolSpec] = tools) -> str:
    selected = choose_tool(query, available_tools)
    if not selected:
        return "Fallback: no hay tool adecuada."
    print(f"[Tool choice] {selected.name}")
    return selected.func(query)


## Sección 3 — Description precisa vs pobre

Una descripción pobre elimina señales para el routing. Por eso diseñar tools también es diseñar prompts.


In [ ]:
print(orchestrator_run("VPN"))
poor_tools = [ToolSpec("HRAgent", hr_agent, "ayuda"), ToolSpec("TechAgent", tech_agent, "ayuda"), ToolSpec("BillingAgent", billing_agent, "ayuda")]
print(orchestrator_run("VPN", poor_tools))


## Checks automáticos

Los checks verifican el contrato mínimo del ejercicio. En Starter pueden fallar hasta completar los TODOs; en Resolution deben pasar.


In [ ]:
def run_checks():
    assert orchestrator_run("VPN").startswith("TechAgent")
    print("Checks E12 OK")
run_checks()


## ¿Qué aprendiste hoy?

- Dejar que el modelo elija tools por descripción.
- Separar responsabilidades vuelve el sistema más auditable.
- Los contratos explícitos hacen que el orquestador dependa menos de texto libre.

## Próximo ejercicio

Continuá con el siguiente notebook de M3L3 para agregar una pieza más de coordinación multiagente.
